# 05 : Transferencia de un codificador GRU y adaptación

**Pregunta:** ¿transferir patrones temporales de otras tiendas mejora el pronóstico
sobre tiendas destino frente a inicializar la misma arquitectura desde cero?

Se usan tiendas 1–36 como origen y 37–54 como destino, sin elegirlas por resultados.
Consulta el protocolo fijado antes de evaluar en `docs/02_method_and_results.md`.
La comparación se limita a las mismas claves destino: no es la tabla del panel completo.
El modo `full` elige CUDA si está disponible; de lo contrario, usa CPU con los mismos
datos y la misma receta. `smoke` verifica ejecución con datos sintéticos.

In [ ]:
# Pinned source setup; Colab preserves its compatible resident libraries.
import importlib.util
import os
import re
import runpy
import subprocess
import sys
import tempfile
from getpass import getpass
from pathlib import Path

# The published academic URL can be supplied through the environment.
REPO_URL = os.environ.get("RETAIL_FORECAST_REPO_URL", "").strip() or globals().get("REPO_URL", "")
if not REPO_URL:
    if (Path.cwd() / "pyproject.toml").is_file():
        REPO_URL = str(Path.cwd())
    else:
        REPO_URL = "https://github.com/camiloandcu/Retail-Demand-Forecasting.git"
if not REPO_URL:
    raise ValueError("Indica la URL del repositorio académico publicado.")
CODE_REVISION = "8353a8d9937e4d8061b026d6885296fcd6b09726"
MODELS = False
if sys.version_info[:2] not in {(3, 10), (3, 11), (3, 12), (3, 13)}:
    raise RuntimeError("Use Python 3.10–3.13 before installing or acquiring data.")
if not re.fullmatch(r"[0-9a-f]{40}", CODE_REVISION):
    raise RuntimeError("CODE_REVISION must be a reviewed full commit SHA.")
IN_COLAB = (
    importlib.util.find_spec("google") is not None
    and importlib.util.find_spec("google.colab") is not None
)
REPO_ROOT = Path(
    os.environ.get(
        "RETAIL_FORECAST_CHECKOUT",
        str(Path("/content" if IN_COLAB else tempfile.gettempdir()) / f"retail-academic-{CODE_REVISION}"),
    )
).absolute()
if REPO_ROOT.is_symlink() or any(p.is_symlink() for p in REPO_ROOT.parents):
    raise RuntimeError("Checkout path must not contain symlinks; use a new directory.")


def checkout_git(*arguments):
    return subprocess.run(
        ["git", "-C", str(REPO_ROOT), *arguments],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()


if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(REPO_ROOT)], check=True)
    # A squash merge can leave the reviewed SHA reachable only by its PR ref.
    subprocess.run(
        ["git", "-C", str(REPO_ROOT), "fetch", "--no-tags", "origin", CODE_REVISION],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO_ROOT), "checkout", "--detach", CODE_REVISION],
        check=True,
    )
# Validate before executing even the shared helper from this checkout.
try:
    valid = (
        checkout_git("rev-parse", "--show-toplevel") == str(REPO_ROOT)
        and checkout_git("remote", "get-url", "origin") == REPO_URL
        and checkout_git("rev-parse", "HEAD") == CODE_REVISION
        and not checkout_git("status", "--porcelain", "--untracked-files=all")
    )
except subprocess.CalledProcessError:
    valid = False
if not valid:
    raise RuntimeError(
        "Dirty, incomplete or mismatched checkout. Preserve it; use a new directory."
    )
helper = runpy.run_path(str(REPO_ROOT / "src/retail_forecast/notebook_runtime.py"))
runtime_evidence = helper["bootstrap"](
    REPO_ROOT, CODE_REVISION, REPO_URL, models=MODELS, colab=IN_COLAB, neural=True
)
print(
    "Notebook revision:",
    os.getenv("RETAIL_FORECAST_NOTEBOOK_REVISION", "unrecorded; retain immutable notebook URL"),
)
WORKSPACE_ROOT = globals().get(
    "WORKSPACE_ROOT",
    Path("/content") if IN_COLAB else (
        Path.cwd() if (Path.cwd() / "pyproject.toml").is_file() else REPO_ROOT
    ),
)
os.chdir(REPO_ROOT)

In [ ]:
from dataclasses import replace

import json

import numpy as np
import pandas as pd
from IPython.display import display

from retail_forecast.acquisition import acquire_competition_data
from retail_forecast.artifacts import read_completed_run
from retail_forecast.config import load_config
from retail_forecast.data import required_files_exist, validate_dataset, write_synthetic_dataset
from retail_forecast.eda import validate_kaggle_hashes
from retail_forecast.pipeline import predict, request
from retail_forecast.transfer import transfer_experiment, transfer_recipe

MODE = os.getenv("RETAIL_FORECAST_MODE", "full")
if MODE not in {"smoke", "full"}:
    raise ValueError("RETAIL_FORECAST_MODE must be smoke or full")
config = load_config(
    REPO_ROOT / ("configs/gru.yaml" if MODE == "full" else "configs/gru-smoke.yaml")
)
config = replace(
    config,
    paths=replace(
        config.paths,
        raw_data_dir=WORKSPACE_ROOT / config.paths.raw_data_dir.relative_to(REPO_ROOT),
        artifacts_dir=WORKSPACE_ROOT / config.paths.artifacts_dir.relative_to(REPO_ROOT),
    ),
)
print("Mode:", MODE, "Policy:", config.data.availability_policy)
print("Effective transfer recipe:", transfer_recipe(config))
print("Runtime identity:", runtime_evidence)

## Datos y separación temporal

Coloca los siete CSV en `data/raw` o proporciona el ZIP mediante el procedimiento
de la guía. La selección de épocas usa C=O−16 y valida C+1…O. Ninguna etiqueta
posterior a O interviene en el entrenamiento. Cada grupo aprende sus propias
transformaciones. Se transfieren exclusivamente los pesos de `model.gru`:
embeddings, cabeza, optimizador y escaladores se inicializan en el destino.

In [ ]:
if config.data.source == "synthetic":
    write_synthetic_dataset(config)
elif not required_files_exist(config.paths.raw_data_dir):
    acquire_competition_data(
        config.paths.raw_data_dir, (WORKSPACE_ROOT / "data", WORKSPACE_ROOT), token_reader=getpass
    )
if MODE == "full":
    validate_kaggle_hashes(config.paths.raw_data_dir)
display(pd.DataFrame([validate_dataset(config).to_dict()]))

## Entrenamiento y evaluación

La función comienza con una prueba de recursos, sin calcular scores de validación.
La proyección de tiempo es informativa: el modo completo no impone un corte por
minutos. En CPU puede tardar más; se mantienen las épocas máximas, la selección
interna y los controles de memoria. La receta completa queda en `manifest.json`, incluidos los hashes de
entrada, la semilla y los grupos. Una ejecución incompleta no es un resultado.
Para cada corte se repite el preentrenamiento con su fecha válida, se seleccionan
épocas internamente y se reajustan modelos nuevos hasta O antes de pronosticar.

In [ ]:
study = transfer_experiment(config)
RUN_DIR = Path(study["run_dir"])
manifest = json.loads((RUN_DIR / "manifest.json").read_text())
assert manifest["status"] == "completed"
print("Evidence directory:", RUN_DIR)
display(pd.DataFrame([{
    "status": manifest["status"], "source": manifest["data_source"],
    "training_seconds": manifest["training_seconds"],
    "wall_seconds": manifest["wall_seconds"],
    "target_rows_per_model": manifest["rows_per_model"],
}]))

## Resultados emparejados

RMSLE agrupado es la medida principal. Un valor menor indica menor error sobre
ventas observadas en estas tiendas y periodos. Se muestran también los tres
cortes y los 16 horizontes: el promedio entre cortes no sustituye al score agrupado.

In [ ]:
metrics = pd.read_csv(RUN_DIR / "results.csv")
summary = metrics[metrics.scope.eq("global")].set_index("model_id")
display(summary)
print("RMSLE adapted − scratch:",
      summary.loc["adapted", "rmsle"] - summary.loc["scratch", "rmsle"])
display(metrics[metrics.scope.eq("fold")].pivot(
    index="segment", columns="model_id", values="rmsle"
))
import matplotlib.pyplot as plt
horizons = metrics[metrics.scope.eq("horizon")].copy()
horizons["segment"] = horizons["segment"].astype(int)
horizons.pivot(index="segment", columns="model_id", values="rmsle").sort_index().plot(
    marker="o", xlabel="Forecast horizon (days)", ylabel="RMSLE"
)
plt.show()

In [ ]:
histories = json.loads((RUN_DIR / "training.json").read_text())
display(pd.DataFrame([{
    "phase": h["phase"], "cutoff": h["cutoff"],
    "epochs": len(h["history"]), "selected_epochs": h["selected_epochs"],
    "training_seconds": h["training_seconds"],
} for h in histories]))
print("Selected epochs:", json.loads((RUN_DIR / "selection.json").read_text()))
print("Probe:", json.loads((RUN_DIR / "feasibility.json").read_text()))

## Qué muestra la ejecución registrada

La adaptación obtuvo RMSLE 0,490339 frente a 0,498156 desde cero sobre 28.512 claves destino por modelo: reducción relativa descriptiva de 1,57 %. Mejoró los tres cortes, pero no todos los horizontes; los días 3, 11, 12 y 13 fueron peores con adaptación. La selección interna eligió épocas adaptada/control 8/5, 6/8 y 8/3. Estos valores corresponden a la ejecución Colab registrada; en `smoke` solo se comprueba funcionamiento.

Se copiaron únicamente pesos del codificador GRU. La cabeza, embeddings, transformaciones y optimizador destino se aprendieron de nuevo, y todas las capas se adaptaron. La información de otras tiendas y su coste de preentrenamiento impiden tratar el experimento como igualdad de datos o cómputo total. Una semilla y tres periodos próximos no permiten afirmar una mejora estable. No se compara este score con el de la GRU global, porque cambian tiendas y receta.